In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.model_selection import LeaveOneOut, cross_val_score, StratifiedKFold
from sklearn.feature_selection import SelectKBest, f_classif, RFECV
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings("ignore")



df = pd.read_csv("Final_data/songs_features.csv")  # <-- replace with your file path

#df = df.drop(columns=['spectral_flatness', 'energy_std', 'mfcc_2_mean', 'mfcc_8_std', 'beat_strength', 'mfcc_1_mean', 'mfcc_11_mean', 'mfcc_1_std', 'chroma_std', 'mfcc_10_std', 'mfcc_9_std', 'mfcc_7_std', 'spectral_rolloff', 'energy_mean', 'mfcc_3_std', 'spectral_centroid', 'chroma_mean', 'dynamic_range', 'mfcc_8_mean'])


feature_cols = [c for c in df.columns if c not in ("name", "label")]
X = df[feature_cols].values
y = df["label"].values

print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Class balance — positive: {y.sum()}, negative: {(y==0).sum()}\n")



rf = RandomForestClassifier(n_estimators=400)
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=feature_cols)
importances = importances.sort_values(ascending=False)

print("\nTop 15 features by importance:")
for feat, score in importances.head(15).items():
    print(f"{feat} {score:.4f}")

# Features to drop — bottom 50% by importance
weak_features = importances.tail(len(importances) // 2).index.tolist()
print(f"\nSuggested features to drop (bottom 50%): {len(weak_features)}")
print(f"{weak_features}\n")


TOP_K_FEATURES = 8

selector = SelectKBest(f_classif, k=TOP_K_FEATURES)
selector.fit(X, y)

selected_mask = selector.get_support()
selected_features = [f for f, s in zip(feature_cols, selected_mask) if s]

scores = pd.Series(selector.scores_, index=feature_cols)
print("\nF-scores for selected features:")
for feat in selected_features:
    print(f"{feat} F={scores[feat]:.2f}")

X_pruned = selector.transform(X)


svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced"))
])

loo = LeaveOneOut()
loo_scores = cross_val_score(svm_pipeline, X_pruned, y, cv=loo, scoring="accuracy")

print(f"LOO accuracy: {loo_scores.mean():.3f} ± {loo_scores.std():.3f}")
print(f"Correct: {loo_scores.sum()} / {len(loo_scores)}")

#Also try with all features for comparison
loo_scores_all = cross_val_score(
    Pipeline([("scaler", StandardScaler()), ("svm", SVC(kernel="rbf", class_weight="balanced"))]),
    X, y, cv=loo, scoring="accuracy"
)
print(f"LOO accuracy ALL:  {loo_scores_all.mean():.3f} ± {loo_scores_all.std():.3f}\n\n\n")

best_c, best_score = None, 0
for C in [0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0]:
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel="rbf", C=C, gamma="scale", class_weight="balanced"))
    ])
    score = cross_val_score(pipe, X_pruned, y, cv=loo, scoring="accuracy").mean()
    print(f"  C={C}  LOO accuracy: {score:.3f}")
    if score > best_score:
        best_score = score
        best_c = C

print(f"\nBest C={best_c}, accuracy={best_score:.3f}")



final_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", C=best_c, gamma="scale",
                class_weight="balanced", probability=True))
])
final_pipeline.fit(X_pruned, y)


Dataset: 35 samples, 38 features
Class balance — positive: 18, negative: 17


Top 15 features by importance:
mfcc_0_std 0.0787
tempo 0.0603
mfcc_0_mean 0.0510
mfcc_6_mean 0.0453
mfcc_9_mean 0.0386
mfcc_7_mean 0.0369
mfcc_6_std 0.0333
zcr_mean 0.0328
mfcc_7_std 0.0307
mfcc_4_std 0.0307
mfcc_3_mean 0.0305
spectral_flatness 0.0294
mfcc_5_std 0.0293
mfcc_5_mean 0.0286
mfcc_12_std 0.0280

Suggested features to drop (bottom 50%): 19
['mfcc_2_std', 'mfcc_8_std', 'energy_std', 'mfcc_2_mean', 'beat_strength', 'mfcc_11_mean', 'energy_mean', 'chroma_mean', 'spectral_rolloff', 'mfcc_1_mean', 'mfcc_10_mean', 'mfcc_8_mean', 'mfcc_4_mean', 'dynamic_range', 'chroma_std', 'mfcc_3_std', 'spectral_centroid', 'mfcc_9_std', 'mfcc_10_std']


F-scores for selected features:
mfcc_0_std F=6.80
mfcc_2_mean F=2.42
mfcc_3_mean F=6.61
mfcc_4_mean F=2.08
mfcc_5_mean F=5.59
mfcc_6_mean F=6.50
mfcc_7_mean F=5.71
mfcc_12_std F=2.41
LOO accuracy:  0.686 ± 0.464
Correct:       24.0 / 35
LOO accuracy ALL:  0.629 ± 0.483


,steps,"[('scaler', ...), ('svm', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,C,0.01
,kernel,'rbf'
,degree,3
,gamma,'scale'


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier  # or RandomForestRegressor
from sklearn.metrics import accuracy_score  # or mean_squared_error
from sklearn.svm import SVC  # or SVR for regression
from sklearn.preprocessing import StandardScaler

In [19]:
df.columns

Index(['name', 'tempo', 'beat_strength', 'energy_mean', 'energy_std',
       'dynamic_range', 'chroma_mean', 'chroma_std', 'spectral_centroid',
       'spectral_contrast', 'spectral_rolloff', 'spectral_flatness',
       'mfcc_0_mean', 'mfcc_0_std', 'mfcc_1_mean', 'mfcc_1_std', 'mfcc_2_mean',
       'mfcc_2_std', 'mfcc_3_mean', 'mfcc_3_std', 'mfcc_4_mean', 'mfcc_4_std',
       'mfcc_5_mean', 'mfcc_5_std', 'mfcc_6_mean', 'mfcc_6_std', 'mfcc_7_mean',
       'mfcc_7_std', 'mfcc_8_mean', 'mfcc_8_std', 'mfcc_9_mean', 'mfcc_9_std',
       'mfcc_10_mean', 'mfcc_10_std', 'mfcc_11_mean', 'mfcc_11_std',
       'mfcc_12_mean', 'mfcc_12_std', 'zcr_mean', 'label'],
      dtype='object')

In [56]:
X = df[selected_features]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25)


scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = SVC(kernel='rbf', C=0.1, gamma='scale')
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Accuracy:", 1 - accuracy_score(y_test, y_pred))


Accuracy: 0.8888888888888888


In [59]:
importances.to_csv("Final_data/feature_importances.csv")

In [60]:
rf

,n_estimators,400
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False
